In [1]:
!pip install yfinance plotly tqdm pandas numpy

In [10]:
import yfinance as yf
import pandas as pd
import numpy as np
from tqdm import tqdm
import plotly.express as px
import plotly.graph_objects as go

# ============================================
# 1. CARREGAR CSV COM AS 200 MAIORES AÇÕES
# ============================================
df_raw = pd.read_csv("top_200_b3_stocks.csv")
tickers = df_raw["symbol"].dropna().tolist()
print(f"Total de tickers carregados: {len(tickers)}")

Total de tickers carregados: 200


In [12]:
def get_data_yahoo(ticker):
    try:
        t = yf.Ticker(ticker)
        info = t.info

        # Campos necessários para nosso modelo adaptado
        ebitda = info.get("ebitda")
        enterprise_value = info.get("enterpriseValue")
        operating_margin = info.get("operatingMargins")
        total_revenue = info.get("totalRevenue")
        market_cap = info.get("marketCap")
        total_debt = info.get("totalDebt")
        total_cash = info.get("totalCash")

        # Só aceitamos tickers com TODOS os dados necessários
        if None in [ebitda, enterprise_value, operating_margin, total_revenue, market_cap, total_debt, total_cash]:
            return None

        # Calcular Operating Income
        operating_income = operating_margin * total_revenue

        # Capital Investido aproximado
        invested_capital = market_cap + total_debt - total_cash
        if invested_capital <= 0:
            return None

        # ROC aproximado
        roc_approx = operating_income / invested_capital

        # Earnings Yield
        ey = ebitda / enterprise_value

        return {
            "ticker": ticker,
            "ebitda": ebitda,
            "enterprise_value": enterprise_value,
            "operating_margin": operating_margin,
            "total_revenue": total_revenue,
            "market_cap": market_cap,
            "total_debt": total_debt,
            "total_cash": total_cash,
            "operating_income": operating_income,
            "invested_capital": invested_capital,
            "roc_approx": roc_approx,
            "ey": ey
        }

    except Exception as e:
        return None


In [13]:
def get_data_yahoo(ticker):
    try:
        t = yf.Ticker(ticker)
        info = t.info

        # Campos necessários para nosso modelo adaptado
        ebitda = info.get("ebitda")
        enterprise_value = info.get("enterpriseValue")
        operating_margin = info.get("operatingMargins")
        total_revenue = info.get("totalRevenue")
        market_cap = info.get("marketCap")
        total_debt = info.get("totalDebt")
        total_cash = info.get("totalCash")

        # Só aceitamos tickers com TODOS os dados necessários
        if None in [ebitda, enterprise_value, operating_margin, total_revenue, market_cap, total_debt, total_cash]:
            return None

        # Calcular Operating Income
        operating_income = operating_margin * total_revenue

        # Capital Investido aproximado
        invested_capital = market_cap + total_debt - total_cash
        if invested_capital <= 0:
            return None

        # ROC aproximado
        roc_approx = operating_income / invested_capital

        # Earnings Yield
        ey = ebitda / enterprise_value

        return {
            "ticker": ticker,
            "ebitda": ebitda,
            "enterprise_value": enterprise_value,
            "operating_margin": operating_margin,
            "total_revenue": total_revenue,
            "market_cap": market_cap,
            "total_debt": total_debt,
            "total_cash": total_cash,
            "operating_income": operating_income,
            "invested_capital": invested_capital,
            "roc_approx": roc_approx,
            "ey": ey
        }

    except Exception as e:
        return None


In [14]:
dados = []
for tk in tqdm(tickers, desc="Coletando dados do Yahoo"):
    info = get_data_yahoo(tk)
    if info is not None:
        dados.append(info)

df = pd.DataFrame(dados)
print("Total de ações com dados completos:", len(df))
df.head()


Coletando dados do Yahoo: 100%|██████████| 200/200 [00:41<00:00,  4.85it/s]

Total de ações com dados completos: 152


,ticker,ebitda,enterprise_value,operating_margin,total_revenue,market_cap,total_debt,total_cash,operating_income,invested_capital,roc_approx,ey
0,PETR4.SA,191036997632,732543975424,0.36224,491446009856,434167447552,376082989056,62001000448,1.780214e+11,748249436160,0.237917,0.260786
1,VALE3.SA,74967998464,388791074816,0.32317,213320007680,301760086016,112818003968,32393000960,6.893863e+10,382185089024,0.180380,0.192823
2,WEGE3.SA,8762923008,187305705472,0.19865,41379594240,189016096768,4617108992,7335310848,8.220056e+09,186297894912,0.044123,0.046784
3,ABEV3.SA,26864080896,204389056512,0.26013,90470252544,220633415680,2896066048,19840921600,2.353403e+10,203688560128,0.115539,0.131436
4,KLBN11.SA,7395928064,54271193088,0.13319,20800421888,128619708416,37574037504,9725681664,2.770408e+09,156468064256,0.017706,0.136277


In [15]:
# Ranking individual
df["rank_roc"] = df["roc_approx"].rank(ascending=False)
df["rank_ey"] = df["ey"].rank(ascending=False)

# Ranking combinado
df["score_magic"] = df["rank_roc"] + df["rank_ey"]

# Ordenar
df_sorted = df.sort_values("score_magic").reset_index(drop=True)

print("Top 15 ações (Magic Fórmula Adaptada):")
df_sorted.head(15)


Top 15 ações (Magic Fórmula Adaptada):


,ticker,ebitda,enterprise_value,operating_margin,total_revenue,market_cap,total_debt,total_cash,operating_income,invested_capital,roc_approx,ey,rank_roc,rank_ey,score_magic
0,PSSA3.SA,5278740992,18530273280,0.11545,40937201664,30939670528,707561984,13114774528,4.726200e+09,18532457984,0.255023,0.284871,3.0,4.0,7.0
1,PETR4.SA,191036997632,732543975424,0.36224,491446009856,434167447552,376082989056,62001000448,1.780214e+11,748249436160,0.237917,0.260786,4.0,8.0,12.0
2,SAPR4.SA,2247791104,3684817920,0.31590,7090981888,11975584768,7255705088,5880009216,2.240041e+09,13351280640,0.167777,0.610014,14.0,1.0,15.0
3,CMIN3.SA,6311273984,26012338176,0.30784,17925320704,30256488448,9357185024,13602187264,5.518131e+09,26011486208,0.212142,0.242626,5.0,12.0,17.0
4,BEEF3.SA,4531187200,18457782272,0.07391,51340525568,6068701184,26719475712,14893214720,3.794578e+09,17894962176,0.212047,0.245489,6.0,11.0,17.0
5,EUCA4.SA,696891008,2505238528,0.14154,3080702976,1748382720,1238204032,340700000,4.360427e+08,2645886752,0.164800,0.278174,15.0,5.0,20.0
6,MYPK3.SA,1438441984,5994540032,0.06296,15756522496,1515670144,5633702912,1624952064,9.920307e+08,5524420992,0.179572,0.239959,11.0,13.0,24.0
7,WHRL3.SA,1372160000,6681693696,0.10361,12881661952,6665993728,2241605120,2161737984,1.334669e+09,6745860864,0.197850,0.205361,7.0,19.0,26.0
8,MOVI3.SA,5144406016,20654084096,0.22680,14261235712,4432501760,19538487296,3316904960,3.234448e+09,20654084096,0.156601,0.249075,21.0,10.0,31.0
9,OPCT3.SA,619129024,2714467584,0.21484,2040610048,1536962944,1823352064,652537024,4.384047e+08,2707777984,0.161906,0.228085,17.0,14.0,31.0


In [16]:
top10 = df_sorted.head(10).copy()
top10


,ticker,ebitda,enterprise_value,operating_margin,total_revenue,market_cap,total_debt,total_cash,operating_income,invested_capital,roc_approx,ey,rank_roc,rank_ey,score_magic
0,PSSA3.SA,5278740992,18530273280,0.11545,40937201664,30939670528,707561984,13114774528,4.726200e+09,18532457984,0.255023,0.284871,3.0,4.0,7.0
1,PETR4.SA,191036997632,732543975424,0.36224,491446009856,434167447552,376082989056,62001000448,1.780214e+11,748249436160,0.237917,0.260786,4.0,8.0,12.0
2,SAPR4.SA,2247791104,3684817920,0.31590,7090981888,11975584768,7255705088,5880009216,2.240041e+09,13351280640,0.167777,0.610014,14.0,1.0,15.0
3,CMIN3.SA,6311273984,26012338176,0.30784,17925320704,30256488448,9357185024,13602187264,5.518131e+09,26011486208,0.212142,0.242626,5.0,12.0,17.0
4,BEEF3.SA,4531187200,18457782272,0.07391,51340525568,6068701184,26719475712,14893214720,3.794578e+09,17894962176,0.212047,0.245489,6.0,11.0,17.0
5,EUCA4.SA,696891008,2505238528,0.14154,3080702976,1748382720,1238204032,340700000,4.360427e+08,2645886752,0.164800,0.278174,15.0,5.0,20.0
6,MYPK3.SA,1438441984,5994540032,0.06296,15756522496,1515670144,5633702912,1624952064,9.920307e+08,5524420992,0.179572,0.239959,11.0,13.0,24.0
7,WHRL3.SA,1372160000,6681693696,0.10361,12881661952,6665993728,2241605120,2161737984,1.334669e+09,6745860864,0.197850,0.205361,7.0,19.0,26.0
8,MOVI3.SA,5144406016,20654084096,0.22680,14261235712,4432501760,19538487296,3316904960,3.234448e+09,20654084096,0.156601,0.249075,21.0,10.0,31.0
9,OPCT3.SA,619129024,2714467584,0.21484,2040610048,1536962944,1823352064,652537024,4.384047e+08,2707777984,0.161906,0.228085,17.0,14.0,31.0


In [17]:
df_plot = df_sorted.copy()
df_plot["grupo"] = "Outras"
df_plot.loc[df_plot["ticker"].isin(top10["ticker"]), "grupo"] = "Top 10"

fig = px.scatter(
    df_plot,
    x="ey",
    y="roc_approx",
    color="grupo",
    hover_data=["ticker", "score_magic", "rank_roc", "rank_ey"],
    title="Magic Fórmula Adaptada — ROC vs EY (Yahoo Finance)",
    labels={"ey": "Earnings Yield (EY)", "roc_approx": "ROC Aproximado"}
)
fig.show()


In [18]:
fig_bar = px.bar(
    top10,
    x="ticker",
    y=["roc_approx", "ey"],
    barmode="group",
    title="Top 10 — ROC (aprox) e EY",
)
fig_bar.show()


In [21]:
# ================================================
# 7. BACKTESTING ROBUSTO E SEM ERROS
# ================================================
import yfinance as yf
import numpy as np
import pandas as pd
import plotly.graph_objects as go

print("Rodando backtest com os tickers:")
print(top10["ticker"].tolist())

tickers_bt = top10["ticker"].tolist()

# ----------------------------------------------------------
# 7.1 Baixar preços ajustados SEM MULTIINDEX
# ----------------------------------------------------------
prices = yf.download(
    tickers_bt,
    period="5y",
    interval="1d",
    auto_adjust=True,
    progress=False
)

# Se vier MultiIndex, pegamos apenas "Close"
if isinstance(prices.columns, pd.MultiIndex):
    prices = prices["Close"]

# Se vier Series, convertendo para DataFrame
if isinstance(prices, pd.Series):
    prices = prices.to_frame()

# Garantir ordenação de colunas e retirar colunas vazias
prices = prices.dropna(axis=1, how="all").sort_index()

print("Colunas finais de preços:", list(prices.columns))
print("Shape:", prices.shape)

# ----------------------------------------------------------
# 7.2 Retornos diários
# ----------------------------------------------------------
returns = prices.pct_change().dropna()

# Evita erro se alguma coluna vier vazia
returns = returns.dropna(axis=1, how="all")

# Montar carteira equal-weight
n = returns.shape[1]
weights = np.array([1/n] * n)

portfolio_returns = (returns * weights).sum(axis=1)

# CURVA ACUMULADA
portfolio_cum = (1 + portfolio_returns).cumprod()
portfolio_cum.name = "Carteira_Magic"

# ----------------------------------------------------------
# 7.3 IBOVESPA (benchmark)
# ----------------------------------------------------------
ibov = yf.download(
    "^BVSP",
    period="5y",
    interval="1d",
    auto_adjust=True,
    progress=False
)

# ibov pode vir Series ou DataFrame
if isinstance(ibov, pd.DataFrame):
    if "Close" in ibov.columns:
        ibov = ibov["Close"]
    else:
        ibov = ibov.iloc[:, 0]

ibov.name = "Ibovespa"

ibov_returns = ibov.pct_change().dropna()
ibov_cum = (1 + ibov_returns).cumprod()
ibov_cum.name = "Ibovespa"

# ----------------------------------------------------------
# 7.4 Alinhar índices e montar DataFrame final
# ----------------------------------------------------------
df_bt = pd.concat([portfolio_cum, ibov_cum], axis=1).dropna()

print(df_bt.head())
print(df_bt.tail())


Rodando backtest com os tickers:
['PSSA3.SA', 'PETR4.SA', 'SAPR4.SA', 'CMIN3.SA', 'BEEF3.SA', 'EUCA4.SA', 'MYPK3.SA', 'WHRL3.SA', 'MOVI3.SA', 'OPCT3.SA']
Colunas finais de preços: ['BEEF3.SA', 'CMIN3.SA', 'EUCA4.SA', 'MOVI3.SA', 'MYPK3.SA', 'OPCT3.SA', 'PETR4.SA', 'PSSA3.SA', 'SAPR4.SA', 'WHRL3.SA']
Shape: (1246, 10)
            Carteira_Magic     ^BVSP
Date                                
2021-02-23        1.017052  1.013591
2021-02-24        1.021348  1.017470
2021-02-25        0.994713  0.987456
2021-02-26        0.983430  0.967919
2021-03-01        0.975047  0.970558
            Carteira_Magic     ^BVSP
Date                                
2025-11-27        1.908320  1.392410
2025-11-28        1.912947  1.399272
2025-12-01        1.909410  1.395216
2025-12-02        1.936495  1.417041
2025-12-03        1.954544  1.422874


In [24]:
# ================================================
# 8. GRÁFICO DO BACKTEST EM % ACUMULADO
# ================================================
import plotly.graph_objects as go

# Converte de "fator" (1.0 → 1.9) para "% acumulado"
# Ex.: 1.40 -> 40%
df_pct = (df_bt - 1.0) * 100

print("Colunas disponíveis em df_pct:", list(df_pct.columns))

fig = go.Figure()

for col in df_pct.columns:
    fig.add_trace(go.Scatter(
        x=df_pct.index,
        y=df_pct[col],
        mode="lines",
        name=col,
        line=dict(width=3)
    ))

titulo_principal = "Backtest – Carteira Magic Adaptada vs Ibovespa"
fig.update_layout(
    title=f"{titulo_principal}<br><sup>Retorno acumulado em % – últimos 5 anos</sup>",
    xaxis_title="Data",
    yaxis_title="Retorno acumulado (%)",
    hovermode="x unified",
    template="plotly_white",
    legend=dict(
        title="Séries",
        orientation="h",
        yanchor="top",
        y=-0.2,
        xanchor="center",
        x=0.5
    )
)

fig.show()


Colunas disponíveis em df_pct: ['Carteira_Magic', '^BVSP']


In [25]:
# ================================================
# 9. TABELA DE RENTABILIDADE MENSAL (IBOV x CARTEIRA)
# ================================================
import pandas as pd

# df_bt: curvas acumuladas
# Converte curva acumulada → preços (índice 1.0 no início)
prices_bt = df_bt.copy()

# Calcula retornos mensais
monthly_returns = prices_bt.resample("M").last().pct_change().dropna()

# Converte para %
monthly_returns = monthly_returns * 100

# Renomeia colunas para ficar mais amigável
monthly_returns.columns = ["Carteira", "Ibovespa"]

# Criar formatador de meses
months = ["Jan", "Fev", "Mar", "Abr", "Mai", "Jun",
          "Jul", "Ago", "Set", "Out", "Nov", "Dez"]

# Montar tabela ano a ano
final_table = {}

for year in sorted(monthly_returns.index.year.unique()):
    df_year = monthly_returns[monthly_returns.index.year == year]

    # novo df com linha da Carteira e Ibovespa
    row_carteira = df_year["Carteira"].reset_index(drop=True)
    row_ibov     = df_year["Ibovespa"].reset_index(drop=True)

    # alinhar tamanho (caso ano atual tenha menos meses)
    max_len = row_carteira.shape[0]
    colnames = months[:max_len]

    table = pd.DataFrame([
        row_carteira.values,
        row_ibov.values
    ], index=["Carteira", "Ibovespa"], columns=colnames)

    final_table[year] = table.round(2)

final_table


/tmp/ipython-input-3109860969.py:11: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.



{2021:            Jan   Fev   Mar   Abr   Mai   Jun   Jul   Ago   Set   Out
 Carteira  3.37  6.31  2.74  1.79 -7.38 -0.23 -3.76 -7.15 -0.18  7.25
 Ibovespa  6.00  1.94  6.16  0.46 -3.94 -2.48 -6.57 -6.74 -1.53  2.85,
 2022:            Jan   Fev   Mar    Abr   Mai    Jun   Jul   Ago   Set   Out   Nov  \
 Carteira  0.98  1.04  2.51  -2.23  3.94 -13.85  3.09  8.21 -5.78  6.63 -3.98   
 Ibovespa  7.22  0.67  6.06 -10.10  3.22 -11.50  4.69  6.16  0.47  5.45 -3.06   
 
            Dez  
 Carteira  0.48  
 Ibovespa -2.18  ,
 2023:            Jan   Fev   Mar   Abr    Mai   Jun   Jul   Ago   Set   Out    Nov  \
 Carteira  9.41 -4.92  0.55 -0.03  12.46  7.76  6.84 -1.51  1.89 -7.90  16.38   
 Ibovespa  3.18 -7.57 -2.91  2.50   3.74  9.00  3.27 -5.09  0.71 -2.93  12.54   
 
            Dez  
 Carteira  7.82  
 Ibovespa  5.38  ,
 2024:            Jan   Fev   Mar   Abr   Mai   Jun   Jul   Ago   Set   Out   Nov  \
 Carteira -5.48 -1.46  1.60 -4.51 -1.89  3.22  1.32  5.71  0.50 -4.39 -1.21   
 Iboves

In [27]:
for year, table in final_table.items():
    # Calculate yearly cumulative returns for the current year
    yearly_monthly_returns = monthly_returns[monthly_returns.index.year == year]
    yearly_cum_carteira_pct = ((1 + yearly_monthly_returns['Carteira'] / 100).prod() - 1) * 100
    yearly_cum_ibovespa_pct = ((1 + yearly_monthly_returns['Ibovespa'] / 100).prod() - 1) * 100

    # Add 'Total' column to the current year's table
    table['Total'] = pd.Series([yearly_cum_carteira_pct, yearly_cum_ibovespa_pct], index=['Carteira', 'Ibovespa']).round(2)

    print(f"\n===== {year} =====")
    display(table)

# Calculate overall cumulative returns for the entire period
overall_cum_carteira = (df_bt['Carteira_Magic'].iloc[-1] - 1) * 100
overall_cum_ibovespa = (df_bt['^BVSP'].iloc[-1] - 1) * 100

print("\n===== Retorno Acumulado Total =====")
overall_summary = pd.DataFrame({
    'Carteira': [f'{overall_cum_carteira:.2f}%'],
    'Ibovespa': [f'{overall_cum_ibovespa:.2f}%']
}, index=['Total'])
display(overall_summary)


===== 2021 =====


,Jan,Fev,Mar,Abr,Mai,Jun,Jul,Ago,Set,Out,Total
Carteira,3.37,6.31,2.74,1.79,-7.38,-0.23,-3.76,-7.15,-0.18,7.25,1.59
Ibovespa,6.00,1.94,6.16,0.46,-3.94,-2.48,-6.57,-6.74,-1.53,2.85,-4.74



===== 2022 =====


,Jan,Fev,Mar,Abr,Mai,Jun,Jul,Ago,Set,Out,Nov,Dez,Total
Carteira,0.98,1.04,2.51,-2.23,3.94,-13.85,3.09,8.21,-5.78,6.63,-3.98,0.48,-0.98
Ibovespa,7.22,0.67,6.06,-10.10,3.22,-11.50,4.69,6.16,0.47,5.45,-3.06,-2.18,4.97



===== 2023 =====


...

,Jan,Fev,Mar,Abr,Mai,Jun,Jul,Ago,Set,Out,Nov,Dez,Total
Carteira,-5.48,-1.46,1.60,-4.51,-1.89,3.22,1.32,5.71,0.50,-4.39,-1.21,-6.47,-12.99
Ibovespa,-4.79,0.99,-0.71,-1.70,-3.04,1.48,3.02,6.54,-3.08,-1.60,-3.12,-4.29,-10.36



===== 2025 =====


,Jan,Fev,Mar,Abr,Mai,Jun,Jul,Ago,Set,Out,Nov,Dez,Total
Carteira,5.24,-4.35,10.45,4.93,6.39,5.68,-3.42,5.02,5.31,-1.69,2.77,2.17,44.63
Ibovespa,4.87,-2.64,6.08,3.69,1.45,1.33,-4.17,6.28,3.40,2.26,6.37,1.69,34.48



===== Retorno Acumulado Total =====


,Carteira,Ibovespa
Total,95.45%,42.29%


In [28]:
# ================================================
# 10. TABELA FINAL – MÉTRICAS COMPARATIVAS
# ================================================
import pandas as pd
import numpy as np

# -------------------------
# Funções auxiliares
# -------------------------

def annual_return(series):
    """Retorno anualizado (CAGR)."""
    total = series.iloc[-1] / series.iloc[0]
    years = (series.index[-1] - series.index[0]).days / 365
    return (total ** (1/years) - 1) * 100

def ytd_return(series):
    """Retorno no ano corrente."""
    df = series[series.index.year == series.index.max().year]
    return (df.iloc[-1] / df.iloc[0] - 1) * 100

def annual_volatility(returns_daily):
    """Volatilidade anualizada."""
    return (returns_daily.std() * np.sqrt(252)) * 100

def max_drawdown(series):
    """Máximo drawdown percentual."""
    roll_max = series.cummax()
    dd = series / roll_max - 1
    return dd.min() * 100

def sharpe_ratio(returns_daily, rf=0.105):   # CDI 10.5% a.a.
    """Sharpe anualizado."""
    mean_r = returns_daily.mean() * 252
    vol = returns_daily.std() * np.sqrt(252)
    return (mean_r - rf) / vol


# -------------------------
# Preparação dos dados
# -------------------------
prices = df_bt.copy()
returns_daily = prices.pct_change().dropna()

# Tabela final
resultados = []

for col in prices.columns:
    serie = prices[col]
    ret_daily = returns_daily[col]

    resultados.append({
        "Série": col,
        "Retorno Anualizado (%)": annual_return(serie),
        "Retorno YTD (%)": ytd_return(serie),
        "Volatilidade (%)": annual_volatility(ret_daily),
        "Máximo Drawdown (%)": max_drawdown(serie),
        "Sharpe Ratio": sharpe_ratio(ret_daily)
    })

tabela_final = pd.DataFrame(resultados)
tabela_final = tabela_final.set_index("Série")
tabela_final = tabela_final.round(2)

tabela_final


,Retorno Anualizado (%),Retorno YTD (%),Volatilidade (%),Máximo Drawdown (%),Sharpe Ratio
Série,,,,,
Carteira_Magic,14.65,45.73,19.48,-23.72,0.27
^BVSP,7.36,34.66,17.43,-26.50,-0.10


In [29]:
# ================================================
# 11. TABELA DE RETORNOS ANUAIS
# ================================================

annual_table = {}

# Converter a curva acumulada para base 1 para cálculo anual
for col in prices.columns:
    series = prices[col]
    df_yearly = series.resample("Y").last()
    yearly_ret = df_yearly.pct_change() * 100
    annual_table[col] = yearly_ret.round(2)

annual_table = pd.DataFrame(annual_table)
annual_table.index = annual_table.index.year  # só ano
annual_table


/tmp/ipython-input-3678810085.py:10: FutureWarning:

'Y' is deprecated and will be removed in a future version, please use 'YE' instead.



,Carteira_Magic,^BVSP
Date,,
2021,NaN,NaN
2022,-0.98,4.97
2023,57.00,21.95
2024,-12.99,-10.36
2025,44.63,34.48
